# N022 · 字符串映射与字典序

**目标：** 用双向映射表达一一对应关系。

**先修：** N006, N007。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 字符映射；双射；字典序；前缀关系；位置编码。

**配套讲解来源：** [同名逐章意见](../../comment/022_string_mapping_order.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决的是"两个字符串是不是同构"的问题。给你两个等长的字符串 s 和 t，问：能不能把 s 里的每个字符固定地替换成另一个字符，从而得到 t？"固定"是关键词——同一个字符每次出现都必须换成同一个目标，不许这次换成 a、下次换成 b。

举个具体到字母的例子：输入 `'egg'` 和 `'add'` → 输出 `True` → 为什么？我们可以让 e→a、g→d，那么 egg 就变成了 add，替换规则前后一致，所以成立。

再看一个最容易被忽视的反例：输入 `'ab'` 和 `'aa'` → 输出 `False` → 为什么？正向看似乎没问题（a→a、b→a，每个源字符的映射都一致），但你倒过来查就会发现：a 和 b 两个源字符都挤着映射到同一个目标 a。这意味着替换规则是"压扁"的、不可逆的——你拿到结果里的 a，根本说不清它原来 是 a 还是 b。这种"只查一边"的检查是本章要治的病。

本章还处理一个变体问题——"单词模式"：把"字符对字符"换成"模式串的字符对句子里的单词"。输入 pattern=`'abba'`、text=`'dog cat cat dog'` → 输出 `True`，因为 a→dog、b→cat 的对应从头到尾一致；而 text=`'dog cat cat fish'` → 输出 `False`，因为第二个 dog 位置的 a 突然要对应 fish，规则前后打架。

两条题本质是同一个问题：判断两个序列之间是否构成一一对应，所以本章用一个公共函数统一解决。

## 2. 基础知识：先读懂这些词

- **字符映射（character mapping）**："某个源字符永远替换成同一个目标字符"的对应规则，比如 e→a。它就像一张密码表：加密时查表替换，而且全篇只用这一张表。
- **正向映射与反向映射（forward / backward mapping）**：正向映射是"源→目标"的表（e→a），反向映射是"目标→源"的表（a→e）。只查正向表能保证"一个源不换多个目标"，只查反向表能保证"多个源不挤向一个目标"。两边一起查，才是一一对应。
- **双射（bijection：一一对应）**：既不"一源多目标"也不"多源一目标"的映射。打个比方：十个人坐十个座位，每人一个座、每个座只有一人，既没有站着的人也没有挤在一起的 人。双射可逆——顺着查和倒着查得到的信息一样多。
- **同构字符串（isomorphic strings）**：两个字符串之间存在一张双射密码表，能把一个整串替换成另一个。"同构"这个词来自数学，意思就是"结构相同"：字符本身不重要，重要的是"谁和谁总在同一位置结构上出现"。
- **单词模式（word pattern）**：同构思想的换皮版本——左序列不再是单个字符，而是按空格切开的单词。判断规则完全不变，只是"字符"升级成了"单词"。
- **zip（拉链函数）**：Python 内置函数，把两个序列像拉拉链一样逐位置配成对，比如 `zip('egg','add')` 依次给出 `('e','a')、('g','d')、('g','d')`。本章用它把两个串按位置对齐着检查。
- **split（切分）**：字符串方法，默认按空白把一句话切成单词列表，比如 `'dog cat cat dog'.split()` 得到 `['dog','cat','cat','dog']`。它是"句子变序列"的桥梁，让单词模式能复用同一套双射检查。

## 3. 思路推导：从小例子开始

- **Step 1：先挡掉长度不等的情况。** 长度都不一样，肯定无法逐位置一一对应，直接返回 False，后面不用看了。比如 `'a'` 对 `'ab'`，一个 1 个字符一个 2 个字符，当场出局。
- **Step 2：准备两张表。** 建一个空字典 forward 记"源→目标"，再建一个空字典 backward 记"目标→源"。为什么是两张而不是一张？因为一张表只能挡住"一源多目标"这一种犯规，挡不住"多源挤一个目标"。
- **Step 3：逐位置配对检查。** 用 zip 把左右序列按位置拉成一对对 (x, y)，对每一对做两问：第一问，x 以前登记过映射吗？登记过且不是 y，说明 x 想换目标了，犯规；第二问，y 以前被谁认领过吗？被认领过且不是 x，说明两个源在挤同一个目标，犯规。两问都通过，就把这对登记进两张表。
- **Step 4：扫完即成功。** 所有位置都通过了检查，说明整张密码表前后自洽且可逆，返回 True。
- **Step 5：变体复用。** 同构直接调用本函数；单词模式先用 split 把句子切成单词列表，再调用同一个函数。核心逻辑只写一遍。

**手算演示一：`'egg'` 对 `'add'`（成功案例）**

| 位置 | 配对 (x,y) | forward 检查 | backward 检查 | 登记后的两张表 |
|---|---|---|---|---|
| 0 | (e,a) | e 未登记，通过 | a 未被认领，通过 | forward={e:a}，backward={a:e} |
| 1 | (g,d) | g 未登记，通过 | d 未被认领，通过 | forward={e:a,g:d}，backward={a:e,d:g} |
| 2 | (g,d) | g 已登记且就是 d，通过 | d 已被 g 认领且就是 g，通过 | 表不变 |

三个位置全部通过，返回 True——这正是 “运行示例”部分 表格里 `egg/add → 是` 的那行。

**手算演示二：`'ab'` 对 `'aa'`（只查正向会漏掉的反例）**

| 位置 | 配对 (x,y) | forward 检查 | backward 检查 | 结果 |
|---|---|---|---|---|
| 0 | (a,a) | a 未登记，通过 | a 未被认领，通过 | 登记 a↔a |
| 1 | (b,a) | b 未登记，通过 | **a 已被 a 认领，不是 b，犯规** | 返回 False |

注意第 1 行：如果只有正向表，b→a 完全合法（b 是第一次出现），程序会错误地放行。是 backward 表里"a 已经名花有主"这条记录把问题抓住了。这就是 “思路推导”部分 推导里说的"`ab→aa` 满足单向一致却破坏可逆性"。

**手算演示三：`'foo'` 对 `'bar'`（正向表抓的犯规）**

| 位置 | 配对 (x,y) | 检查结果 |
|---|---|---|
| 0 | (f,b) | 通过，登记 f↔b |
| 1 | (o,a) | 通过，登记 o↔a |
| 2 | (o,r) | forward 里 o 已经登记为 a，现在却要换成 r，**犯规**，返回 False |

这个例子反过来证明：两张表各抓各的犯规，缺一不可。“运行示例”部分 表格里还有 `'paper'` 对 `'title'`，你可以自己按上面格式推一遍，四对 (p,t)(a,i)(p,t)(e,l) 依次登记，最后 (r,e) 也顺利登记，返回 True。

**手算演示（补全）：`'paper'` 对 `'title'`（重复字符的复用场景）**

| 位置 | 配对 (x,y) | 检查要点 | 登记后的 forward / backward |
|---|---|---|---|
| 0 | (p,t) | 双方都未登记，通过 | {p:t} / {t:p} |
| 1 | (a,i) | 双方都未登记，通过 | {p:t,a:i} / {t:p,i:a} |
| 2 | (p,t) | p 已登记且就是 t，纯核对 | 表不变 |
| 3 | (e,l) | 双方都未登记，通过 | {p:t,a:i,e:l} / {t:p,i:a,l:e} |
| 4 | (r,e) | 注意 e 此前是"源"，这里 e 作为"目标"出现，backward 里 e 未被认领，通过 | {p:t,a:i,e:l,r:e} / {t:p,i:a,l:e,e:r} |

这张表值得多看一眼：p 在位置 2 的"纯核对"和 e 在位置 4 的"角色切换"（一会儿当源、一会儿当目标）都不会引起误判，因为两张表查的是"记录是否存在且一致"，而不是"这个字符出现过没有"。

**手算演示四：单词模式 `'abba'` 对 `'dog cat cat dog'`。** split 之后左序列是 `['a','b','b','a']`，右序列是 `['dog','cat','cat','dog']`。配对依次是 (a,dog)(b,cat)(b,cat)(a,dog)：第一对登记 a↔dog，第二对登记 b↔cat，第三、四对是纯重复核对，全部通过，返回 True。若右序列是 `'dog cat cat fish'`，第四对变成 (a,fish)，而 forward 里 a 已登记为 dog，当场犯规返回 False。

## 4. 核心代码：is_isomorphic

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def is_isomorphic(s, t):
    return bijective_match(s, t)
```

### 逐段读懂代码

### 函数 1：`bijective_match`（公共核心：判断双射）

先贴 notebook “分段实现”部分 里的真实代码：

```python
def bijective_match(left, right):
    if len(left) != len(right):
        return False
    forward, backward = ({}, {})
    for x, y in zip(left, right):
        if x in forward and forward[x] != y:
            return False
        if y in backward and backward[y] != x:
            return False
        forward[x] = y
        backward[y] = x
    return True
```

- **第 2 行 `if len(left)!=len(right): return False`**：这是进门第一道闸门。左右序列长度不同就无法逐位置配对（同构要求位置一一对齐），直接判负。测试里的 `'a'` 对 `'ab'` 就是被这行挡下的。顺带一提，对单词模式来说，`len` 比的是"单词个数"而不是"字符个数"，因为传进来的是 split 之后的列表。
- **第 3 行 `forward,backward={},{}`**：一行创建两个空字典——forward 是"源→目标"表，backward 是"目标→源"表。两张表在后面同步更新，互为镜像。
- **第 4 行 `for x,y in zip(left,right)`**：zip 把两个序列按位置拉成一对对元素。循环每一轮拿到同一位置上的源元素 x 和目标元素 y，比如处理 `'egg'`/`'add'` 时第一轮 x='e'、y='a'。
- **第 5 行 `if x in forward and forward[x]!=y: return False`**：第一问——"x 是不是想换目标？"`x in forward` 查 x 是否登记过（字典的成员检查，平均常数时间）；登记过（条件前半为真）但登记的值不是现在的 y（后半也为真），说明同一个源要映射到两个不同目标，犯规，立即返回 False。演示三里 `'foo'` 的第二个 o 就是被这行抓的。
- **第 6 行 `if y in backward and backward[y]!=x: return False`**：第二问——"y 是不是已经被别人占了？"`y in backward` 查 y 是否被认领过；被认领过但主人不是 x，说明两个源在挤同一个目标，犯规返回 False。演示二里 `'ab'→'aa'` 的第二个 a 就是被这行抓的。注意第 5、6 行的条件都写成"两段用 and 连接"的紧凑形式：前半是"有旧记录"，后半是"旧记录对不上"，两者同时成立才算犯规——如果 x 没登记过，前半为假，整句直接短路跳过，安全放行。
- **第 7 行 `forward[x]=y; backward[y]=x`**：两问都通过了，就把这对关系同时写进两张表。用分号把两条赋值写在一行，这是紧凑写法，效果和分两行写完全一样。已有的重复配对（比如 egg 里第二次出现 g→d）会原地重写成相同的值，不影响结果。
- **第 8 行 `return True`**：循环走完都没犯规，说明每个位置都和已有映射兼容，两张表构成完整的双向密码表，返回 True。

### 函数 2：`is_isomorphic`（同构字符串，薄封装）

```python
def is_isomorphic(s, t):
    return bijective_match(s, t)
```

这个函数只有一行：把参数原样转交给 `bijective_match`。它存在的意义是给 LeetCode 205 题提供题目要求的函数名和参数形式（两个字符串 s、t），真正的逻辑全在公共函数里。字符串本身就可以被 zip 和 len 处理，所以不需要任何预处理。

### 函数 3：`word_pattern`（单词模式，加一步切分）

```python
def word_pattern(pattern, text):
    return bijective_match(pattern, text.split())
```

左边的 pattern 本来就是序列（字符序列），右边传入的是一个完整句子字符串，所以先用 `text.split()` 按空格把它切成单词列表，再交给同一个双射判断函数。比如 `'dog cat cat dog'` 切成 `['dog','cat','cat','dog']` 之后，"字符对字符"的检查就原封不动地变成了"字符对单词"的检查——这正是 “思路推导”部分 说的"单词模式只是把字符换成以空格分开的词"。

## 5. 分段实现：按顺序运行

**本章接口：** `is_isomorphic(s, t)`、`word_pattern(pattern, text)`

### bijective_match

In [1]:
def bijective_match(left, right):
    if len(left) != len(right):
        return False
    forward, backward = ({}, {})
    for x, y in zip(left, right):
        if x in forward and forward[x] != y:
            return False
        if y in backward and backward[y] != x:
            return False
        forward[x] = y
        backward[y] = x
    return True

### is_isomorphic

In [2]:
def is_isomorphic(s, t):
    return bijective_match(s, t)

### word_pattern

In [3]:
def word_pattern(pattern, text):
    return bijective_match(pattern, text.split())

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['左序列','右序列','是否双射'],[(s,t,is_isomorphic(s,t)) for s,t in [('egg','add'),('foo','bar'),('ab','aa'),('paper','title')]])

左序列,右序列,是否双射
egg,add,True
foo,bar,False
ab,aa,False
paper,title,True


## 7. 正确性、适用条件与复杂度

每步只接受与已有双向映射兼容的字符对，保持局部双射。扫描结束后映射覆盖所有出现的字符，因此给出全局一一对应。

**代价：** 平均 O(n) 时间、O(u) 映射空间；split 还需与文本长度成正比的存储。

### 展开理解

**为什么是对的？** 我们要说明的是：函数返回 True 时两串确实一一对应，返回 False 时确实不一一对应，两头都 成立。

先看返回 True 的情况。整个扫描过程中，两张表始终保持着"每条已登记的关系都两边唯一"的状态：每次登记前我们都确认过"x 以前若登记过只能是 y"和"y 以前若被认领过只能是 x"，所以任何时刻表里都不会出现一源多目标或多源一目标。循环结束时，两个串里出现过的每一对字符都被检查并登记过了，也就是说表覆盖了全部出现的字符。于是这张表就是一张完整的、两边都不冲突的替换密码表，顺着用能把 left 变成 right，倒着用也能无歧义地变回去，这正是"一一对应"的含义。

再看返回 False 的情况。返回只发生在两种时刻：要么长度不等——长度不等的两个序列在数学上就定义不出逐位置的一一对应；要么在某一对 (x,y) 上查出了冲突——要么 x 此前已经确定映射到别的字符（说明同一个源字符被要求换两个目标），要么 y 已经被别的源字符认领（说明两个源字符被要求挤向同一个目标）。两种冲突都直接违背"一一对应"的定义，所以判负是对的。因为算法只在确凿冲突时才说不、只在全程无冲突时才说是，所以它给出的答案和"是否同构"这个判断标准完全吻合。

**复杂度是多少？** 时间上是平均 O(n)，n 是序列长度（单词模式下同时受字符数和单词数约束，以较长者计）。zip 产生的每一对只被访问一次，每对上的两问都是字典的查询和写入，平均每次只花常数时间，所以 n 个位置总共大约 n 次常数操作。举个具体的数：两个串各 10⁵ 个字符时，大约就是二十来万次字典读写，一瞬间就算完。空间上是 O(u)，u 是出现过的不同字符（或不同单词）的个数——两张表里每个不同的源和目标各存一条，u 最多等于 n，比如 26 个小写字母的场景里 u≤26，两张表加起来撑死几十条记录。另外 notebook 的 “正确性与复杂度”部分 特别提醒：`text.split()` 会把整句话切成一个新列表，这份临时存储和文本长度成正比，是一份额外的空间开销，虽然用完即丢，但记账时不能漏掉它。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言一行行看：

- `assert is_isomorphic('egg','add')`：正常成功值。e→a、g→d 全程一致且双向唯一，测的是"标准同构能被正确认出"。
- `assert not is_isomorphic('foo','bar')`：正向冲突的失败值。第二个 o 已经登记为 a，却要换成 r，测的是 forward 表那行检查真的能拦住"一源换两个目标"。
- `assert not is_isomorphic('ab','aa')`：反向冲突的失败值，也是全章最关键的一条。a→a、b→a 每个源各自一致，单向检查会放行，但两个源挤同一个目标 a 破坏了可逆性，测的是 backward 表那行检查不可缺席。
- `assert not is_isomorphic('a','ab')`：边界值。长度 1 对长度 2，测的是进门那道长度闸门——连位置都配不齐就别谈映射了。
- `assert word_pattern('abba','dog cat cat dog')`：单词模式的正常成功值。a→dog、b→cat 前后一致，测的是 split 之后整套逻辑对单词照样成立。
- `assert not word_pattern('abba','dog cat cat fish')`：单词模式的失败值。同一个 a 前三次对应 dog，最后一次却要对应 fish，测的是模式串字符与单词的对应被打破时能被识破。

这六条正好覆盖了成功、两类冲突、长度边界、单词模式的成败各一条，是一组小而全的契约测试。最后一行 `print('N022: 所有本章断言通过')` 只在全部断言都活下来之后才会执行——assert 一旦失败程序当场报错，所以你看到这行输出就说明上面六条全过了。

In [5]:
assert is_isomorphic('egg', 'add')

assert not is_isomorphic('foo', 'bar')

assert not is_isomorphic('ab', 'aa')

assert not is_isomorphic('a', 'ab')

assert word_pattern('abba', 'dog cat cat dog')

assert not word_pattern('abba', 'dog cat cat fish')

print('N022: 所有本章断言通过')

N022: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 解释单向映射不足。

**练习 2：** 构造两个源字符映射到同一目标的反例。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（解释单向映射为什么不够）**：方向是做一个对照实验。你写一个只有 forward 表的"半吊子版本"，然后喂给它 `'ab'` 和 `'aa'`——它会错误地说 True。你要在纸上把两个位置的登记过程列出来（可以照搬本文第三节演示二的表格格式），指明第 2 个位置上 a 已被认领、单向版却看不见这一层。写清楚输入、期望输出（False）、单向版的实际输出（True），再用本章的 `bijective_match` 验证正确答案。边界可以顺带考虑空串对空串、单字符对单字符这类平凡情形。
- **练习 2（构造两个源字符挤向同一目标的反例）**：这类反例的特征是"右串里某个字符出现次数少于左串中它对应的源字符种类数"。最短的就是 `'ab'` 对 `'aa'`；你不妨再构造一个更长的，比如左右都取 4 个字符，让左串含两个不同字符、右串只含一个重复字符。同样先手算列出每一步两张表的状态，说明在哪一对上 backward 表报错，然后用 `bijective_match` 跑一遍核对。提示：构造时盯住"右串的去重字符个数小于左串的去重字符个数"这个信号，必出反向冲突。

两道练习都不需要写很多代码，重点是把"哪一步、哪张表、抓到了什么"讲给别人听——能讲清楚，才算真懂了双射为什么需要两个方向。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def bijective_match(left, right):
    if len(left) != len(right):
        return False
    forward, backward = ({}, {})
    for x, y in zip(left, right):
        if x in forward and forward[x] != y:
            return False
        if y in backward and backward[y] != x:
            return False
        forward[x] = y
        backward[y] = x
    return True

def is_isomorphic(s, t):
    return bijective_match(s, t)

def word_pattern(pattern, text):
    return bijective_match(pattern, text.split())

# 示例与回归测试
assert is_isomorphic('egg', 'add')

assert not is_isomorphic('foo', 'bar')

assert not is_isomorphic('ab', 'aa')

assert not is_isomorphic('a', 'ab')

assert word_pattern('abba', 'dog cat cat dog')

assert not word_pattern('abba', 'dog cat cat fish')

print('N022: 所有本章断言通过')

N022: 所有本章断言通过


## 11. 代表题与迁移

- **205. Isomorphic Strings（同构字符串）**：这题就是本章的 `is_isomorphic` 本尊，练的是"两张表同时查正向与反向映射"这个核心点，也是检验你能不能想到 `ab→aa` 这类单向检查会漏掉的反例。
- **290. Word Pattern（单词模式）**：这题就是 `word_pattern` 本尊，练的是"把同构思想迁移到单词粒度"——先 split 再复用同一个双射判断，体会"序列元素是字符还是单词，算法结构完全不变"。

这两题在 notebook 里被标注为 canonical（与本章概念一一对应的代表题）。做完它们可以再自问一句：如果把"双向唯一"放宽成"单向一致"，两题的答案分别会怎么错？想明白这一点，本章就没白学。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。